# Manuel de test manuel : Module `kadi.weather`

Bienvenue dans le guide d’utilisation du module météo de KadiPy. Ce module a été pensé avec une approche **"offline-first"**. Il récupère les données via l’API Open-Meteo et les stocke localement dans une base SQLite (`~/.kadi/cache.db`). Ainsi, les calculs peuvent être effectués même sans connexion internet, une fois les données rapatriées.

Nous allons passer en revue chaque fonction, son fonctionnement et comment elle peut aider une coopérative agricole ou une startup AgriTech.

In [1]:
# Importations nécessaires
from kadi.cache import init_db
from kadi.weather import (
    forecast, 
    historical, 
    growing_degree_days, 
    rain_probability, 
    drought_index
)

# 1. Initialisation de la base de données (essentiel au premier lancement)
init_db()
print("Base de données SQLite initialisée et prête !")

Base de données SQLite initialisée et prête !


## 1. Prévisions Météo : `forecast()`

La fonction `forecast` permet d’obtenir les prévisions journalières (températures min/max/moyenne, précipitations) pour une ville donnée sur un horizon défini.

**Paramètres :**
- `location` (str) : Le nom de la ville (ex: "Cotonou", "Parakou").
- `days` (int) : Le nombre de jours de prévision souhaité (par défaut 7).
- `refresh` (bool) : Si True, force une nouvelle requête API en ignorant le cache (utile si on veut absolument des données fraîches).

**Détails Techniques :**
L’appel vérifie d’abord le cache. Si le cache a moins de 24h (défini dans `kadi/config.py`), il est retourné instantanément sans réseau.

In [2]:
# Récupérer les prévisions sur 5 jours pour Cotonou
previsions = forecast("Cotonou", days=5, refresh=False)

print("Source des données :", previsions["data_source"])
print("Jours de prévision obtenus :", len(previsions["data"]))
print("\nAperçu du premier jour :")
print(previsions["data"][0])

Source des données : open-meteo
Jours de prévision obtenus : 5

Aperçu du premier jour :
{'date': '2026-06-28', 'hour': None, 'temperature_min': 25.4, 'temperature_max': 27.3, 'temperature_avg': 26.35, 'precipitation': 4.8, 'data_type': 'forecast', 'data_source': 'open-meteo', 'confidence': 0.95}


## 2. Historique Météorologique : `historical()`

La fonction `historical` récupère l’historique des conditions météo. Ces données passées sont vitales pour calculer des indicateurs agronomiques (voir plus bas) ou analyser des corrélations de rendement.

**Paramètres :**
- `location` (str) : La ville cible.
- `months_back` (int) : Nombre de mois d’historique à récupérer (par défaut 12).
- `metric` (str) : Actuellement ignoré, prévu pour ne filtrer qu’une métrique précise plus tard.

L’historique étant invariant, il est très fortement mis en cache (1 an de durée de vie).

In [5]:
# Récupérer 2 mois d'historique pour Cotonou
historique = historical("Cotonou", days_back= 1)

print(f"Historique récupéré : {len(historique['data'])} jours d'observations.")
print("Premier jour en archive :", historique["data"][0]["date"])

Historique récupéré : 2 jours d'observations.
Premier jour en archive : 2026-06-27


## 3. Degrés-Jours de Croissance (DJC / GDD) : `growing_degree_days()`

L'une des forces de KadiPy ! Les Degrés-Jours de Croissance (Growing Degree Days) mesurent l’accumulation de chaleur pour estimer le stade de développement d’une plante (levée, floraison, maturité).

**Formule utilisée :** `GDD = max(0, Temp_moyenne - Temp_base)`

**Paramètres :**
- `location` (str) : La ville (les données doivent déjà être en cache via l'historique).
- `crop` (str) : La culture (ex: "maïs", "manioc", "riz"). La température de base est automatiquement choisie selon la culture (ex: 10°C pour le maïs, 14°C pour le manioc).
- `start_date` / `end_date` (str) : Période d’évaluation.

In [39]:
start = '2026-05-28'
end = '2026-06-27'

# Calcul pour du Maïs sur les 30 derniers jours
gdd_result = growing_degree_days(
    location="Cotonou", 
    crop="maïs", 
    start_date=start, 
    end_date=end
)

print(f"Plante étudiée : {gdd_result['crop']} (Temp. base: {gdd_result['base_temp']}°C)")
print(f"GDD Accumulés  : {gdd_result['gdd']} degrés-jours")
print(f"Jours valides  : {gdd_result['valid_days']}")

Plante étudiée : maïs (Temp. base: 10.0°C)
GDD Accumulés  : 2533.6 degrés-jours
Jours valides  : 146


## 4. Probabilité de Pluie (Farmer-Friendly) : `rain_probability()`

Pour une vulgarisation simple auprès des agriculteurs (via SMS ou USSD par exemple), cette fonction génère un résumé lisible sur les jours à venir.

**Paramètres :**
- `location` (str) : La ville.
- `days_ahead` (int) : Nombre de jours dans le futur (défaut 3).

In [8]:
# Probabilité de pluie à Cotonou dans les 4 prochains jours
pluie = rain_probability("Cotonou", days_ahead=4) 
"""Il trouve toujours 100 % de probabilité pour la pluie
Le days_ahead > 19 provoque une erreur
"""

print("Message prêt à être envoyé :")
print("--->", pluie["message"])
print("\nDétails techniques :")
print(f"- Niveau de risque : {pluie['risk_level']} (Indice: {pluie['risk_index_pct']}/100)")
print(f"- Quantité prévue : {pluie['total_expected_rain_mm']} mm")

Message prêt à être envoyé :
---> Risque de pluie fort (56 mm attendus sur la période).

Détails techniques :
- Niveau de risque : Fort (Indice: 75/100)
- Quantité prévue : 55.8 mm


## 5. Indice de Sécheresse : `drought_index()`

Dans cette version V1 simplifiée, cet indicateur donne une idée rapide du niveau de précipitation sur les X derniers mois, afin de détecter une éventuelle période de déficit hydrique.

**Paramètres :**
- `location` (str) : La ville.
- `months_back` (int) : Nombre de mois d'historique à évaluer.

In [10]:
# Évaluation de la sécheresse à Abomey sur les 3 derniers mois
secheresse = drought_index("Abomey", months_back=3)

print(f"Mois évalués : {secheresse['months_evaluated']}")
print(f"Précipitations totales : {secheresse['total_precipitation']} mm")
print(f"Moyenne journalière    : {secheresse['avg_precipitation_per_day']} mm/jour")

Mois évalués : 3
Précipitations totales : 334.2 mm
Moyenne journalière    : 3.63 mm/jour
